In [1]:
import os

os.environ.setdefault("SPARK_VERSION", "3.5")

import pandas as pd

from workflow_sifta import optimization_cost
from workflow_sifta._0_splits import DATASETS, SPLIT_SEEDS
from workflow_sifta._4_scoring import outcomes
from workflow_prismadv.results.metrics import dataset_summary, summary

In [2]:
PROGRAMS = {"stats_novelty": "stats-novelty", "gepa_script": "GEPA[S]", "gepa_column": "GEPA[P]",
            "prismadv": "Manual", "sifta": "SIFTA"}
SCENARIOS = {"test_1_cross_new_data": "New Data Batches", "test_2_new_script_train_data": "New Tasks"}
LABELS = {"IPL_win_prediction": "IPL"}

out = outcomes(DATASETS, SPLIT_SEEDS, list(PROGRAMS))
ds, s = dataset_summary(out), summary(out)

In [3]:
def cell(t, key):
    if key not in t.index:
        return "–"
    return f"{100 * t.loc[key, 'f1_mean']:.2f} ± {100 * t.loc[key, 'f1_sd']:.1f}"


rows = {}
for test, scenario in SCENARIOS.items():
    for d in DATASETS:
        t = ds.xs((test, d), level=("benchmark", "dataset"))
        rows[(scenario, LABELS.get(d, d))] = {name: cell(t, p) for p, name in PROGRAMS.items()}
    rows[(scenario, "Mean")] = {name: cell(s.xs(test, level="benchmark"), p) for p, name in PROGRAMS.items()}

# Table 3: macro F1 per target scenario, mean ± SD over the three train/test splits
pd.DataFrame(rows).T

stats-novelty       GEPA[S]       GEPA[P]  \
New Data Batches students       47.49 ± 4.6  62.31 ± 19.3  69.68 ± 12.2   
                 hr_analytics   41.21 ± 3.2   56.55 ± 4.6   56.70 ± 1.2   
                 sleep_health   52.77 ± 2.3   77.95 ± 6.9   69.43 ± 1.5   
                 IPL            55.82 ± 3.5  62.50 ± 23.1   63.76 ± 3.0   
                 imdb           42.65 ± 5.2  63.15 ± 12.3   67.42 ± 2.3   
                 Mean           47.99 ± 0.5  64.49 ± 10.3   65.40 ± 3.1   
New Tasks        students                 –   47.12 ± 7.5   65.34 ± 7.6   
                 hr_analytics             –   62.37 ± 7.9   62.21 ± 5.1   
                 sleep_health             –  66.22 ± 12.2   73.80 ± 6.1   
                 IPL                      –  69.25 ± 20.0   66.40 ± 5.0   
                 imdb                     –  64.33 ± 12.4   68.08 ± 7.8   
                 Mean                     –   61.86 ± 3.1   67.17 ± 3.8   

                                     Manual         SIFTA  
New Data Batches students      57.36 ± 13.1   66.82 ± 5.4  
                 hr_analytics   42.73 ± 8.3   57.85 ± 4.6  
                 sleep_health   70.40 ± 8.6   75.40 ± 2.4  
                 IPL            55.28 ± 3.7   74.95 ± 6.7  
                 imdb           65.50 ± 5.9   74.16 ± 5.5  
                 Mean           58.25 ± 6.6   69.84 ± 4.4  
New Tasks        students       61.52 ± 4.2  70.14 ± 11.6  
                 hr_analytics  45.20 ± 16.1   63.06 ± 8.1  
                 sleep_health  61.07 ± 18.2   72.34 ± 9.8  
                 IPL            51.55 ± 8.5  73.29 ± 11.2  
                 imdb          70.33 ± 12.8   81.23 ± 3.0  
                 Mean           57.93 ± 8.9   72.01 ± 1.2

In [4]:
cost = optimization_cost.summary(optimization_cost.records())


def cell(row, field, scale, digits):
    return f"{row[f'{field}_mean'] / scale:.{digits}f} ± {row[f'{field}_sd'] / scale:.{digits}f}"


# Table G: optimization cost per run (tokens in millions), mean ± SD over the three splits
pd.DataFrame({name: {"Input": cell(cost.loc[p], "input", 1e6, 2), "Output": cell(cost.loc[p], "output", 1e6, 2),
                     "Reasoning": cell(cost.loc[p], "reasoning", 1e6, 2), "LLM calls": cell(cost.loc[p], "calls", 1, 2),
                     "Cost (USD)": cell(cost.loc[p], "usd", 1, 2)}
              for p, name in optimization_cost.PROGRAMS.items()}).T

,Input,Output,Reasoning,LLM calls,Cost (USD)
GEPA[P],4.40 ± 0.16,0.18 ± 0.01,2.63 ± 0.19,566.07 ± 93.90,7.24 ± 1.15
GEPA[S],0.67 ± 0.01,0.16 ± 0.01,0.18 ± 0.02,43.87 ± 2.53,1.18 ± 0.11
SIFTA,7.78 ± 0.24,0.28 ± 0.01,3.51 ± 0.09,798.47 ± 185.40,8.80 ± 2.19
